In [0]:
from pyspark.sql import functions as F

In [0]:
#Constants
SOURCE_NAME = "cdc_records"
TABLE_NAME = "orders_cdc_raw"
SCHEMA = "bronze"
INGEST_CATALOG = "sl_ingest"

In [0]:
#Configs
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())
ENV = configs.get('env', 'dev')
INITIAL_RUN = configs.get('initial_run', False)

INGEST_VOLUME   = f"/Volumes/{INGEST_CATALOG}/{ENV}/landing"

CATALOG        = f"sl_{ENV}"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

print(f"ENV={ENV} | source schema: {INGEST_CATALOG}.{ENV} | ingest_volume: {INGEST_VOLUME} | catalog: {CATALOG} | checkpoints: {CHECKPOINT_BASE}")

ENV=dev | source schema: sl_ingest.dev | ingest_volume: /Volumes/sl_ingest/dev/landing | catalog: sl_dev | checkpoints: /Volumes/sl_dev/bronze/checkpoints


In [0]:
source_path = f"{INGEST_VOLUME}/{SOURCE_NAME}"
target_table = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
checkpoint_path = f"{CHECKPOINT_BASE}/{TABLE_NAME}/"
print(source_path, target_table, checkpoint_path)

In [ ]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
    )

In [0]:
query = (df.writeStream
    .trigger(availableNow=True)
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
     .option("mergeSchema", "true")
    .outputMode("append")
    .toTable(target_table))

query.awaitTermination()

In [ ]:
if INITIAL_RUN: #this is not an ideal production pattern but more straight forward for this used case
    spark.sql(f"ALTER TABLE {target_table} CLUSTER BY (_ingested_at)")